In [1]:
from Crypto.Util.Padding import pad, unpad
import Crypto.Cipher.AES as AES
import secrets
import os

FLAG = os.getenv("FLAG", "Alpaca{dummy}")
key = secrets.token_bytes(16)


def encrypt(plaintext):
    cipher = AES.new(key=key, mode=AES.MODE_CBC)
    encrypted_flag = cipher.encrypt(pad(plaintext.encode(), 16))
    return cipher.iv + encrypted_flag


def decrypt(iv, ciphertext):
    cipher = AES.new(key=key, mode=AES.MODE_CBC, iv=iv)
    a = cipher.decrypt(ciphertext)
    # print("[debug]", a[-16:], file=sys.stderr) # for debug. this is not respond to client. Let's look this output in your environment.
    try:
        unpad(a, 16)
        return True
    except:
        return False


# plaintext = ???????????????A???????????????L???????????????P......\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10|
#             |---16 bytes---||---16 bytes---||---16 bytes---|---...| You can ignore this block                                     |
plaintext = ""
for c in FLAG:
    plaintext += "?" * 15 + c

iv_ciphertext = encrypt(plaintext)
print(f"iv_ciphertext={iv_ciphertext.hex()}")

iv_ciphertext=94d4b82ccf8ae2bc8233bf1eb83d317308c7f7065651cfc25f4f63d83c07f079d88f77e9245851fc1d8e8643ef5d4e39475fe324a0e477bbcb40953d0c7a28d0cd233b406d9c2936717f80d62f4c6c567eae49d630b52433dec44a55b0030fcb307ecca4d23b7e428022dfb562e20d2e594703606d26e3685c328aaf60f25c6822fe9b83e9de60fcf70753a402fa0b7c03c0544a7cb4b0f1382fdf11d92aaa26acc281ed79959e32e78ae2f82fcd16ba818fd303b8b5b0998becaee579147cdae7b009e9e6ab66355d4cb3dfc194c223caf8c44bb05414372d954006be89f9e1695abcd1d75855f92c8af5bab96fe924


In [2]:
bs = 16
bc = len(iv_ciphertext) // bs
blocks = [iv_ciphertext[i * bs : (i + 1) * bs] for i in range(bc)]
iv = blocks[0]
cipher_blocks = blocks[1:]

In [ ]:
def decrypt_block(Ci_1: bytes, Ci: bytes):
    data = bytearray([0] * 16)
    for i in reversed(range(16)):
        pad = 16 - i
        for b in range(256):
            data[i] = b
            if decrypt(bytes(data), Ci):
                print(f"Found byte: data[{i}] = {b^pad^Ci_1[i]:02x}")
                for j in range(i, 16):
                    data[j] ^= pad ^ (pad + 1)
                break
    for j in range(16):
        data[j] ^= Ci_1[j] ^ (pad + 1)
    return data[:16]

In [34]:
decrypt_block(cipher_blocks[11],cipher_blocks[12])

Found byte: data[15] = 7d
Found byte: data[14] = 3f
Found byte: data[13] = 3f
Found byte: data[12] = 3f
Found byte: data[11] = 3f
Found byte: data[10] = 3f
Found byte: data[9] = 3f
Found byte: data[8] = 3f
Found byte: data[7] = 3f
Found byte: data[6] = 3f
Found byte: data[5] = 3f
Found byte: data[4] = 3f
Found byte: data[3] = 3f
Found byte: data[2] = 3f
Found byte: data[1] = 3f
Found byte: data[0] = 3f


bytearray(b'???????????????}')